# 02 — Semantic-ID cluster & hierarchy visualisation

Inspect the learned RQ-VAE codebook: do level-1 codes form coherent module-level
clusters, and do a class's methods share a prefix? 

Run first:
```bash
python -m src.semantic_ids.assign_ids configs/experiments/semid_qlora.yaml
```

In [ ]:
import json, sys, pathlib, numpy as np
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from src.utils import load_config
from src.semantic_ids.assign_ids import run_pipeline

cfg = load_config('../configs/experiments/semid_qlora.yaml')
cfg['paths']['target_repo'] = '../data/target_repo'
entities, emb, model, vocab, assignments = run_pipeline(cfg)
codes = np.array([assignments[e.uid]['codes'] for e in entities])
print(emb.shape, codes.shape, 'codebook usage:', model.codebook_usage())

In [ ]:
# 2-D projection of entity embeddings coloured by level-1 code
import matplotlib.pyplot as plt
try:
    import umap
    xy = umap.UMAP(n_neighbors=15, random_state=42).fit_transform(emb)
except Exception:
    from sklearn.decomposition import PCA
    xy = PCA(n_components=2).fit_transform(emb)
plt.figure(figsize=(9, 7))
plt.scatter(xy[:, 0], xy[:, 1], c=codes[:, 0], cmap='tab20', s=8)
plt.title('Entity embeddings coloured by L1 semantic-ID code'); plt.colorbar(label='L1 code')

In [ ]:
# Hierarchy tree: a sampled L1 -> L2 -> entities structure
from collections import defaultdict
tree = defaultdict(lambda: defaultdict(list))
for e in entities:
    c = assignments[e.uid]['codes']
    tree[c[0]][c[1]].append(e.name)
for l1 in sorted(tree)[:3]:
    print(f'L1={l1}')
    for l2 in sorted(tree[l1])[:3]:
        print(f'  L2={l2}: {tree[l1][l2][:5]}')